## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: klue/bert-base  
데이터셋: klue/ynat  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.


데이터 불러오기

In [ ]:
from datasets import load_dataset
code
#VSC-0be14aa1
python
import numpy as np
try:
    from sklearn.metrics import classification_report, confusion_matrix
    use_sklearn = True
except Exception:
    use_sklearn = False

# 예측 수행
preds_output = trainer.predict(test_dataset)
logits = preds_output.predictions
labels = preds_output.label_ids
preds = np.argmax(logits, axis=-1)

if use_sklearn:
    report = classification_report(labels, preds, zero_division=0, digits=4)
    cm = confusion_matrix(labels, preds)
    print('--- Classification Report ---')
    print(report)
    print('--- Confusion Matrix ---')
    print(cm)
else:
    acc = float((preds == labels).mean())
    print(f'Accuracy: {acc:.4f}')


# --- Quick imbalance handling demo: class-weighted loss using a small quick retrain ---
import torch
import torch.nn as nn
from collections import Counter
import numpy as np

# Compute class counts from training labels (assumes train_labels list exists for local fallback)
num_classes = int(model.config.num_labels)
try:
    counts = [0]*num_classes
    if 'train_labels' in globals():
        for l in train_labels: counts[int(l)] += 1
    else:
        # fallback: estimate from train_dataset if possible
        for i in range(len(train_dataset)): counts[int(train_dataset[i]['labels'])] += 1
except Exception:
    counts = [1]*num_classes
counts = np.array(counts, dtype=np.float32)
counts = np.where(counts==0, 1.0, counts)
class_weights = torch.tensor(1.0 / counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * num_classes
print('class counts:', counts.tolist())
print('class weights:', class_weights.tolist())

# Subclass Trainer to apply weighted CrossEntropyLoss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get('labels')
        # forward without labels to get logits
        outputs = model(**{k:v for k,v in inputs.items() if k != 'labels'})
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Quick retrain for 1 epoch to demonstrate effect
from transformers import TrainingArguments as _TA
quick_args = _TA(output_dir='./result/weighted_quick', num_train_epochs=1, per_device_train_batch_size=2, per_device_eval_batch_size=2, learning_rate=2e-5)
w_trainer = WeightedTrainer(model=model, args=quick_args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=data_collator, compute_metrics=compute_metrics)
w_trainer.train()
w_metrics = w_trainer.evaluate()
print('Weighted-trainer metrics:', w_metrics)
with open('./result/weighted_eval.txt', 'w', encoding='utf-8') as f:
    f.write(str(w_metrics))
# 결과 저장
with open('./result/eval_detailed.txt', 'w', encoding='utf-8') as f:
    if use_sklearn:
        f.write('Classification report:
        f.write(report)
        f.write('
        f.write(str(cm))
    else:
        f.write(f'Accuracy: {acc:.4f}

# 샘플 예측 저장(테스트셋 첫 50개)
import csv
with open('./result/test_predictions.csv', 'w', encoding='utf-8', newline='') as csvf:
    writer = csv.writer(csvf)
    writer.writerow(['text','label','pred'])
    # test_texts may be list when using local fallback
    texts = test_texts if ('test_texts' in globals() and isinstance(test_texts, list)) else []
print('Saved ./result/eval_detailed.txt and ./result/test_predictions.csv')        writer.writerow([t, int(labels[i]), int(preds[i])])        t = texts[i] if i < len(texts) else ''    for i in range(min(len(preds), 50)):    texts = test_texts if ('test_texts' in globals() and isinstance(test_texts, list)) else [t for t in getattr(test_dataset, '__len__', lambda:0)()][:50]    # test_texts may be list when using local fallback    writer.writerow(['text','label','pred'])    writer = csv.writer(csvf)with open('./result/test_predictions.csv', 'w', encoding='utf-8', newline='') as csvf:import csv# 샘플 예측 저장(테스트셋 첫 50개)')        f.write(f'Accuracy: {acc:.4f}    else:        f.write(str(cm))')Confusion matrix:        f.write('        f.write(report)')        f.write('Classification report:    if use_sklearn:with open('./result/eval_detailed.txt', 'w', encoding='utf-8') as f:# 결과 저장    print(f'Accuracy: {acc:.4f}')    acc = float((preds == labels).mean())else:    print(cm)    print('--- Confusion Matrix ---')    print(report)    print('--- Classification Report ---')    cm = confusion_matrix(labels, preds)    report = classification_report(labels, preds, zero_division=0, digits=4)if use_sklearn:preds = np.argmax(logits, axis=-1)labels = preds_output.label_idslogits = preds_output.predictionspreds_output = trainer.predict(test_dataset)# 예측 수행    use_sklearn = Falseexcept Exception:    use_sklearn = True    from sklearn.metrics import classification_report, confusion_matrixtry:import numpy as nppython#VSC-0be14aa1codeprint(inf('신제품 출시로 시장 기대감 상승'))inf = pipeline('text-classification', model='./result/best_model', tokenizer='./result/best_model')from transformers import pipelinepython#VSC-23826e03codetokenizer.save_pretrained('./result/best_model')trainer.save_model('./result/best_model')# 저장print(metrics)metrics = trainer.evaluate()print(train_result)train_result = trainer.train())    compute_metrics=compute_metrics    data_collator=data_collator,    eval_dataset=eval_dataset,    train_dataset=train_dataset,    args=training_args,    model=model,trainer = Trainer(python#VSC-e7e4f95ecode)    logging_dir='./logs',    weight_decay=0.01,    learning_rate=2e-5,    per_device_eval_batch_size=2,    per_device_train_batch_size=2,    num_train_epochs=3,    output_dir='./result',training_args = TrainingArguments(python#VSC-0806f3c4code    return {'accuracy': acc, 'precision': float(prec), 'recall': float(rec), 'f1': float(f1s)}        prec = rec = f1s = 0.0    except Exception:        prec, rec, f1s, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)        from sklearn.metrics import precision_recall_fscore_support    try:    acc = float((preds == labels).astype(float).mean())    preds = np.argmax(logits, axis=-1)    logits, labels = eval_preddef compute_metrics(eval_pred):from transformers import TrainingArguments, Trainer, EarlyStoppingCallbackimport numpy as nppython#VSC-97672488code토큰화 및 데이터 준비는 위의 코드 셀에서 처리됩니다. 로컬 폴백일 경우 Torch Dataset 래퍼가 생성됩니다.markdown#VSC-c31a1e02markdown    test_dataset = tokenized_dataset['test'] if 'test' in tokenized_dataset else tokenized_dataset['validation']    eval_dataset = tokenized_dataset['validation'] if 'validation' in tokenized_dataset else tokenized_dataset['test']    train_dataset = tokenized_dataset['train']        tokenized_dataset = tokenized_dataset.map(lambda x: {'labels': x[label_src]}, batched=True)    if label_src != 'labels':    label_src = label_candidates[0]        raise ValueError('No label column found in dataset')    if len(label_candidates) == 0:    label_candidates = [c for c in dataset['train'].column_names if c in ['label','labels','target','category']]    tokenized_dataset = dataset.map(lambda ex: tokenize_function(ex, text_col), batched=True)    print('Using text column:', text_col)        text_col = dataset['train'].column_names[0]    if text_col is None:            break            text_col = c        if c in ['title','text','document','headline','article']:    for c in dataset['train'].column_names:    text_col = Noneelse:    print('Created local Torch Datasets')    tokenized_dataset = None    test_dataset = NewsDataset(test_texts, test_labels, tokenizer)    eval_dataset = NewsDataset(val_texts, val_labels, tokenizer)    train_dataset = NewsDataset(train_texts, train_labels, tokenizer)            return len(self.labels)        def __len__(self):            return item            item['labels'] = torch.tensor(self.labels[idx])            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}        def __getitem__(self, idx):            self.labels = labels            self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_length)        def __init__(self, texts, labels, tokenizer, max_length=128):    class NewsDataset(TorchDataset):    from torch.utils.data import Dataset as TorchDatasetif 'is_local' in globals() and is_local:    return tokenizer(examples[text_column], truncation=True, max_length=128)def tokenize_function(examples, text_column):import torchpython#VSC-9c99fa8bcodedata_collator = DataCollatorWithPadding(tokenizer=tokenizer)from transformers import DataCollatorWithPadding# Data collator는 패딩을 배치 단위로 처리합니다)    num_labels=7    model_name,model = AutoModelForSequenceClassification.from_pretrained(tokenizer = AutoTokenizer.from_pretrained(model_name)model_name = "klue/bert-base"python#VSC-6ae4d8e9code기본 모델과 토크나이저 불러오기markdown#VSC-bb5cb1bfmarkdownprint('Saved ./result/eval_detailed.txt and ./result/test_predictions.csv')        writer.writerow([t, int(labels[i]), int(preds[i])])        t = texts[i] if i < len(texts) else ''    for i in range(min(len(preds), 50)):if is_local:
    print(f'train size: {len(train_texts)}, val size: {len(val_texts)}, test size: {len(test_texts)}')
else:
    print(dataset)

SyntaxError: unterminated string literal (detected at line 77) (1624706615.py, line 77)

In [22]:
# LR sweep quick test: increase learning rate and run 2 epochs
new_lr = 5e-5
from transformers import TrainingArguments, Trainer

print('Testing learning rate =', new_lr)
new_args = TrainingArguments(output_dir='./result/lr_test', num_train_epochs=2, per_device_train_batch_size=2, per_device_eval_batch_size=2, learning_rate=new_lr)
new_trainer = Trainer(model=model, args=new_args, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=data_collator, compute_metrics=compute_metrics)
new_train_out = new_trainer.train()
print('Train output:', new_train_out)
new_metrics = new_trainer.evaluate()
print('LR test metrics:', new_metrics)
new_trainer.save_model('./result/lr_test_model')
tokenizer.save_pretrained('./result/lr_test_model')

Testing learning rate = 5e-05


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.72it/s]


Train output: TrainOutput(global_step=10, training_loss=1.1969749450683593, metrics={'train_runtime': 8.0836, 'train_samples_per_second': 2.474, 'train_steps_per_second': 1.237, 'total_flos': 1315614336000.0, 'train_loss': 1.1969749450683593, 'epoch': 2.0})


c:\Users\LRO\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,1.640595,10,0.000000,0.000000,0.000000,0.000000


LR test metrics: {'eval_loss': 1.6405949592590332, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]


('./result/lr_test_model\\tokenizer_config.json',
 './result/lr_test_model\\tokenizer.json')